# Isolated Temporal Evidence: Selected Temporal Representation

This notebook is the **final isolated temporal-evidence evaluation reported in the thesis** for distinguishing `NORMAL` conversations from controlled `LAG` anomalies.

It follows directly from the preceding **Full Temporal Development Pipeline** notebook.

## From temporal development to the selected representation

The development pipeline explored a richer temporal evidence packet containing:

- cleaned / filtered participant turn sequences,
- overlap information,
- signed local A-end-to-B-start response offsets,
- deterministic summary statistics over those offsets,
- global alignment-shift features,
- frozen temporal reference profiles estimated from a separate NORMAL reference pool.

That development stage established the complete temporal workflow: VAD cleaning, eligibility filtering, frozen-reference construction, synthetic lag generation, local temporal statistics, global alignment features, and targeted Qwen2.5-Omni reasoning.

For the final isolated temporal experiment, the model-facing representation was intentionally reduced.

The selected temporal representation retains:

### Local temporal evidence
- the complete signed strict-offset list,
- number of valid offsets,
- mean,
- median,
- maximum,
- 75th percentile,
- 90th percentile,
- number of offsets above 1.5 seconds,
- percentage of offsets above 1.5 seconds.

### Global temporal evidence
- best B correction shift,
- estimated B lateness,
- alignment-score gain relative to zero shift,
- number of bilateral events,
- event coverage.

The model does **not** receive:

- filtered turn sequences,
- overlap features,
- participation `speaks` indicators,
- semantic summaries,
- conversation identifiers,
- gold labels.

This reduction is important because filtered turn lists and overlap were useful during temporal development but were not retained in the selected temporal representation used for the reported isolated result.

## Upstream dataset construction

The source interactions and temporal references were constructed in the preceding temporal-development pipeline using the common dataset-selection procedure:

- VAD regions separated by at most **0.75 s** were merged;
- each participant was required to contribute at least **5 speaking turns**;
- each participant was required to contribute at least **10 seconds of speech**;
- this retained **167 eligible source conversations**;
- **50 conversations** were reserved for frozen temporal-reference estimation;
- the development pool supplied the cases used for this isolated temporal evaluation.

This notebook does not repeat that upstream construction. Instead, it loads the already prepared frozen reference artifacts and the final structured case database so that the selected temporal representation can be evaluated without regenerating the source cases.

## Exact evaluation set

The notebook selects, without resampling:

- **100 NORMAL cases**,
- **50 LAG +2 s cases**,
- **50 LAG +3 s cases**.

This gives exactly **200 evaluation cases**:

- 100 NORMAL,
- 100 LAG.

## Thesis-reported result

The selected temporal representation achieves:

| Case | Correct | Total | Accuracy |
|---|---:|---:|---:|
| NORMAL | 83 | 100 | 83.0% |
| LAG +2 s | 41 | 50 | 82.0% |
| LAG +3 s | 47 | 50 | 94.0% |
| **Overall** | **171** | **200** | **85.5%** |

The resulting binary confusion matrix is:

```text
             Pred NORMAL   Pred LAG
Gold NORMAL       83          17
Gold LAG          12          88
```

This is the **isolated temporal evidence result reported in the thesis**.

> **Note on legacy naming.** Some code variables, output directories, console messages, and cached artifact names still contain the original development label `Diagnostic 1`. These identifiers are intentionally preserved so that the executed code and stored outputs remain exactly identical to the experiment that produced the reported result.

## 1. Install Dependencies

Install the exact Qwen2.5-Omni environment used for the selected-representation evaluation.

Run this cell once in a fresh Colab runtime. Restart the session after installation, then continue.

In [ ]:
# Run this installation cell ONCE in a fresh Colab runtime.

!pip uninstall -y transformers
!pip install -q "transformers==4.57.6" accelerate bitsandbytes sentencepiece modelscope
!pip install -q qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

print(
    "Installation complete. Select Runtime -> Restart session, "
    "then continue from the next cell."
)

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 17.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━

## 2. Mount Google Drive and Configure the Evaluation

This notebook reuses the artifacts created during temporal development rather than reconstructing the temporal dataset from raw participant records.

It loads:

- the frozen temporal reference statistics,
- the final structured case database,
- the local Qwen2.5-Omni checkpoint,
- the persistent output directory used by the original run.

The paths retain their original diagnostic naming to preserve exact reproducibility.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

import copy
import hashlib
import json
import math
import os
import re
import tempfile

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm


PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)

REFERENCE_BASE_STATS_PATH = (
    PROJECT_DIR
    / "frozen_reference_base_statistics.json"
)

REFERENCE_SHIFT_STATS_PATH = (
    PROJECT_DIR
    / "frozen_reference_global_shift_statistics.json"
)

FINAL_DATABASE_PATH = (
    PROJECT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)

DIAGNOSTIC_DIR = (
    PROJECT_DIR
    / "diagnostic1_targeted_lag_R1_temporal_features"
)

DIAGNOSTIC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CASE_MANIFEST_PATH = (
    DIAGNOSTIC_DIR
    / "exact_100_normal_100_lag_case_manifest.json"
)

PROMPT_TEMPLATE_PATH = (
    DIAGNOSTIC_DIR
    / "prompt_template.txt"
)

RESULTS_PATH = (
    DIAGNOSTIC_DIR
    / "results_cache.json"
)

RESULTS_CSV_PATH = (
    DIAGNOSTIC_DIR
    / "results.csv"
)

MAX_NEW_TOKENS = 260

PILOT_MAX_CASES = None

DIAGNOSTIC_PROMPT_VERSION = (
    "diagnostic1_targeted_normal_vs_lag23_"
    "R1_offsets_all_global_no_turns_no_overlap_v1"
)


required_paths = {
    "Project directory": PROJECT_DIR,
    "Frozen base statistics": REFERENCE_BASE_STATS_PATH,
    "Frozen global-shift statistics": REFERENCE_SHIFT_STATS_PATH,
    "Final 400-case database": FINAL_DATABASE_PATH,
    "Local Qwen checkpoint": MODEL_PATH,
}


print("=" * 96)
print("ARTIFACT PATH AUDIT")
print("=" * 96)

for name, path in required_paths.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists())


assert PROJECT_DIR.exists(), (
    f"Project directory not found: {PROJECT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    f"Frozen base statistics not found: {REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    f"Frozen global-shift statistics not found: {REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    f"Final consolidation database not found: {FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    f"Local Qwen checkpoint not found: {MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Project directory: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec
  exists: True
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the Frozen NORMAL, LAG +2 s, and LAG +3 s Reference Profiles

The frozen references were constructed upstream and remain fixed during this evaluation.

Only the reference statistics corresponding to the **selected temporal representation** are exposed to the model:

- local signed-offset distribution statistics,
- the five retained global alignment-shift feature families.

Frozen overlap statistics are deliberately excluded because overlap is not part of the selected model-facing representation.

In [ ]:
def load_json(
    path,
):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


frozen_base_records = load_json(
    REFERENCE_BASE_STATS_PATH
)

frozen_shift_records = load_json(
    REFERENCE_SHIFT_STATS_PATH
)


assert isinstance(
    frozen_base_records,
    list,
)

assert isinstance(
    frozen_shift_records,
    list,
)


PROFILE_ORDER = [
    "NORMAL",
    "LAG_2",
    "LAG_3",
]


base_by_profile = {
    str(record["reference_profile"]): copy.deepcopy(record)
    for record in frozen_base_records
}


shift_by_profile = {
    str(record["reference_profile"]): copy.deepcopy(record)
    for record in frozen_shift_records
}


for profile in PROFILE_ORDER:
    assert profile in base_by_profile, (
        f"Missing frozen base profile: {profile}"
    )

    assert profile in shift_by_profile, (
        f"Missing frozen shift profile: {profile}"
    )


BASE_REFERENCE_FIELDS = [
    "num_offsets",
    "offset_mean",
    "offset_median",
    "offset_max",
    "offset_p75",
    "offset_p90",
    "percent_above_1_5",
]


SHIFT_REFERENCE_FIELDS = [
    "best_B_correction_shift_seconds__mean",
    "best_B_correction_shift_seconds__median",
    "estimated_B_lateness_seconds__mean",
    "estimated_B_lateness_seconds__median",
    "alignment_score_gain_vs_zero__mean",
    "alignment_score_gain_vs_zero__median",
    "best_num_bilateral_events__mean",
    "best_event_coverage__mean",
]


def require_finite_number(
    record,
    field,
    context,
):
    assert field in record, (
        f"Missing {field} in {context}"
    )

    value = float(
        record[field]
    )

    assert math.isfinite(value), (
        f"Non-finite {field} in {context}: {value}"
    )

    return value


selected_base_references = {}

selected_shift_references = {}


for profile in PROFILE_ORDER:
    selected_base_references[profile] = {
        field: require_finite_number(
            base_by_profile[profile],
            field,
            f"base/{profile}",
        )
        for field in BASE_REFERENCE_FIELDS
    }

    selected_shift_references[profile] = {
        field: require_finite_number(
            shift_by_profile[profile],
            field,
            f"shift/{profile}",
        )
        for field in SHIFT_REFERENCE_FIELDS
    }


# Strict exclusion of overlap references.
reference_text_audit = json.dumps(
    {
        "base": selected_base_references,
        "shift": selected_shift_references,
    },
    sort_keys=True,
)

assert "overlap" not in reference_text_audit.lower()


print("=" * 96)
print("SELECTED FROZEN REFERENCE PROFILES")
print("=" * 96)

print("Profiles:", PROFILE_ORDER)

print("\nLOCAL OFFSET-DISTRIBUTION REFERENCES")

display(
    pd.DataFrame.from_dict(
        selected_base_references,
        orient="index",
    )
    .rename_axis("reference_profile")
    .round(4)
)

print("\nGLOBAL ALIGNMENT-SHIFT REFERENCES")

display(
    pd.DataFrame.from_dict(
        selected_shift_references,
        orient="index",
    )
    .rename_axis("reference_profile")
    .round(4)
)

SELECTED FROZEN REFERENCE PROFILES
Profiles: ['NORMAL', 'LAG_2', 'LAG_3']

LOCAL OFFSET-DISTRIBUTION REFERENCES


,num_offsets,offset_mean,offset_median,offset_max,offset_p75,offset_p90,percent_above_1_5
reference_profile,,,,,,,
NORMAL,5.76,0.3021,0.2619,1.0545,0.5902,0.8253,6.734
LAG_2,4.48,1.3327,1.2978,2.5055,1.8657,2.2280,51.149
LAG_3,4.46,1.5862,1.5163,3.0669,2.1840,2.7123,43.525



GLOBAL ALIGNMENT-SHIFT REFERENCES


,best_B_correction_shift_seconds__mean,best_B_correction_shift_seconds__median,estimated_B_lateness_seconds__mean,estimated_B_lateness_seconds__median,alignment_score_gain_vs_zero__mean,alignment_score_gain_vs_zero__median,best_num_bilateral_events__mean,best_event_coverage__mean
reference_profile,,,,,,,,
NORMAL,-0.230,-0.0,0.500,0.0,0.0234,0.0091,11.94,0.5952
LAG_2,-1.100,-1.9,1.668,1.9,0.1881,0.1846,10.78,0.5618
LAG_3,-2.034,-2.9,2.464,2.9,0.2065,0.1792,10.62,0.5511


## 4. Select the Exact 200 Evaluation Cases

No new conversations are sampled and no new lag variants are generated in this notebook.

The experiment selects the existing cases directly from the structured database:

- 100 `normal`,
- 50 `lag_2sec`,
- 50 `lag_3sec`.

This ensures that the reported evaluation is performed on the exact preconstructed cases used in the original run. An immutable manifest is also saved for auditing.

In [ ]:
consolidation_cases = load_json(
    FINAL_DATABASE_PATH
)


assert isinstance(
    consolidation_cases,
    list,
)

assert len(
    consolidation_cases
) == 400


all_case_ids = [
    str(case["case_id"])
    for case in consolidation_cases
]


assert len(
    all_case_ids
) == len(
    set(all_case_ids)
)


def normalize_case_variant(
    value,
):
    return (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


TARGET_VARIANTS = {
    "normal",
    "lag_2sec",
    "lag_3sec",
}


diagnostic_cases = [
    copy.deepcopy(case)
    for case in consolidation_cases
    if normalize_case_variant(
        case.get("case_variant")
    ) in TARGET_VARIANTS
]


variant_counts = Counter(
    normalize_case_variant(
        case.get("case_variant")
    )
    for case in diagnostic_cases
)


expected_variant_counts = {
    "normal": 100,
    "lag_2sec": 50,
    "lag_3sec": 50,
}


assert dict(
    variant_counts
) == expected_variant_counts, (
    "The final database does not contain the expected exact "
    f"diagnostic cases. Found: {dict(variant_counts)}"
)


diagnostic_case_ids = [
    str(case["case_id"])
    for case in diagnostic_cases
]


assert len(
    diagnostic_case_ids
) == 200

assert len(
    diagnostic_case_ids
) == len(
    set(diagnostic_case_ids)
)


for case in diagnostic_cases:
    variant = normalize_case_variant(
        case["case_variant"]
    )

    if variant == "normal":
        expected_gold = "NORMAL"

    else:
        expected_gold = "ANOMALOUS"

    assert (
        str(case["gold_binary_label"]).strip().upper()
        == expected_gold
    ), (
        f"Unexpected gold label for {case['case_id']}"
    )


# Preserve the exact database order and save the immutable manifest.
case_manifest = [
    {
        "case_id": str(case["case_id"]),
        "case_variant": normalize_case_variant(
            case["case_variant"]
        ),
        "gold_binary_label": str(
            case["gold_binary_label"]
        ).strip().upper(),
    }
    for case in diagnostic_cases
]


CASE_MANIFEST_PATH.write_text(
    json.dumps(
        case_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 96)
print("EXACT DIAGNOSTIC CASE AUDIT")
print("=" * 96)

print("Cases selected directly from final database:", len(diagnostic_cases))
print("Variant counts:", dict(variant_counts))
print("Unique case IDs:", len(set(diagnostic_case_ids)))
print("Saved immutable manifest:", CASE_MANIFEST_PATH)

display(
    pd.DataFrame(
        case_manifest
    )
    .groupby(
        [
            "case_variant",
            "gold_binary_label",
        ],
        as_index=False,
    )
    .size()
)

EXACT DIAGNOSTIC CASE AUDIT
Cases selected directly from final database: 200
Variant counts: {'normal': 100, 'lag_2sec': 50, 'lag_3sec': 50}
Unique case IDs: 200
Saved immutable manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/diagnostic1_targeted_lag_R1_temporal_features/exact_100_normal_100_lag_case_manifest.json


,case_variant,gold_binary_label,size
0,lag_2sec,ANOMALOUS,50
1,lag_3sec,ANOMALOUS,50
2,normal,NORMAL,100


## 5. Project the Selected Temporal Evidence

This step constructs the exact evidence packet shown to the reasoner.

The model-facing payload contains only:

### Local temporal fields
- `signed_strict_offsets_seconds`
- `num_signed_strict_offsets`
- `offset_mean_seconds`
- `offset_median_seconds`
- `offset_max_seconds`
- `offset_p75_seconds`
- `offset_p90_seconds`
- `num_offsets_above_1_5_seconds`
- `percent_offsets_above_1_5_seconds`

### Global temporal fields
- `best_B_correction_shift_seconds`
- `estimated_B_lateness_seconds`
- `alignment_score_gain_vs_zero`
- `best_num_bilateral_events`
- `best_event_coverage_percent`

`best_event_coverage` is converted from a stored fraction to a percentage exactly as in the original Structured R1 representation.

The evidence audit explicitly verifies that **filtered turns, overlap, participation fields, and semantic summaries are absent**.

In [ ]:
LOCAL_OFFSET_FIELDS = [
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_SOURCE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


GLOBAL_MODEL_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage_percent",
]


FORBIDDEN_MODEL_EVIDENCE_KEYS = [
    "filtered_turns",
    "participant_A_filtered_turns",
    "participant_B_filtered_turns",
    "clean_overlap_seconds",
    "clean_overlap_percent",
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",
    "speaks",
    "case_id",
    "case_variant",
    "gold_binary_label",
    "gold_anomaly_type",
    "conversation_id",
    "participant_id",
    "lag_seconds",
    "reference_profile",
]


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )

    missing = [
        field
        for field in fields
        if field not in record
    ]

    assert not missing, (
        f"Missing fields in {context}: {missing}"
    )

    return {
        field: copy.deepcopy(
            record[field]
        )
        for field in fields
    }


def build_diagnostic_model_input(
    case,
):
    local_features = select_exact_fields(
        case["local_temporal_features"],
        LOCAL_OFFSET_FIELDS,
        "local_temporal_features",
    )

    global_raw = select_exact_fields(
        case["global_shift_features"],
        GLOBAL_SOURCE_FIELDS,
        "global_shift_features",
    )

    event_coverage_fraction = float(
        global_raw.pop(
            "best_event_coverage"
        )
    )

    global_features = {
        **global_raw,
        "best_event_coverage_percent": round(
            100.0 * event_coverage_fraction,
            6,
        ),
    }

    payload = {
        "analysis_duration_seconds": float(
            case["analysis_duration_seconds"]
        ),
        "local_temporal_features": local_features,
        "global_shift_features": global_features,
    }

    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "local_temporal_features",
        "global_shift_features",
    }

    assert list(
        payload["local_temporal_features"].keys()
    ) == LOCAL_OFFSET_FIELDS

    assert list(
        payload["global_shift_features"].keys()
    ) == GLOBAL_MODEL_FIELDS

    payload_text = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
    ).lower()

    for forbidden in FORBIDDEN_MODEL_EVIDENCE_KEYS:
        assert forbidden.lower() not in payload_text, (
            f"Forbidden model-facing evidence leaked: {forbidden}"
        )

    return payload


# Audit every one of the exact 200 cases before inference.
for case in diagnostic_cases:
    build_diagnostic_model_input(
        case
    )


print("=" * 96)
print("MODEL-FACING EVIDENCE AUDIT PASSED")
print("=" * 96)

print("Cases audited:", len(diagnostic_cases))
print("Local fields:", LOCAL_OFFSET_FIELDS)
print("Global fields:", GLOBAL_MODEL_FIELDS)
print("Filtered turns supplied: False")
print("Overlap supplied: False")
print("Participant speaks supplied: False")
print("Semantic summaries supplied: False")

MODEL-FACING EVIDENCE AUDIT PASSED
Cases audited: 200
Local fields: ['signed_strict_offsets_seconds', 'num_signed_strict_offsets', 'offset_mean_seconds', 'offset_median_seconds', 'offset_max_seconds', 'offset_p75_seconds', 'offset_p90_seconds', 'num_offsets_above_1_5_seconds', 'percent_offsets_above_1_5_seconds']
Global fields: ['best_B_correction_shift_seconds', 'estimated_B_lateness_seconds', 'alignment_score_gain_vs_zero', 'best_num_bilateral_events', 'best_event_coverage_percent']
Filtered turns supplied: False
Overlap supplied: False
Participant speaks supplied: False
Semantic summaries supplied: False


## 6. Build the Targeted NORMAL-vs-LAG Temporal Prompt

The prompt performs a binary temporal-coordination assessment using only the selected temporal features and the frozen reference profiles.

Its interpretation follows the targeted lag-classification logic developed in the preceding temporal pipeline, while instructions referring to filtered turns or overlap are absent because those fields are no longer supplied.

The objective is therefore narrow and controlled: determine whether the retained local and global temporal evidence is sufficient to distinguish `NORMAL` from `LAG`.

In [ ]:
import json
import math


# ============================================================
# DIAGNOSTIC 1 — ORIGINAL LAG PROMPT, SURGICALLY REDUCED
#
# Removed only:
# - Participant A/B filtered turns
# - TURN FORMAT section
# - filtered overlap evidence and rules
# - LAG +1 reference profile
#
# Retained:
# - exact local offset logic
# - NORMAL / LAG +2 / LAG +3 references
# - all five global temporal features
# - original NORMAL vs LAG decision logic
# ============================================================

DIAGNOSTIC_PROMPT_VERSION = (
    "diagnostic1_original_lag_prompt_"
    "R1_offsets_all_global_no_turns_no_overlap_v2"
)


PROFILE_TITLES = {
    "NORMAL": "Typical NORMAL cases",
    "LAG_2": "Typical LAG +2 second cases",
    "LAG_3": "Typical LAG +3 second cases",
}


# ============================================================
# REFERENCE PROFILE TEXT
# ============================================================

def build_local_reference_section(
    profile,
):
    values = selected_base_references[
        profile
    ]

    return f"""
{PROFILE_TITLES[profile]}:

- Number of signed offsets around {values["num_offsets"]:.2f}.
- Mean signed offset around {values["offset_mean"]:.2f} seconds.
- Median signed offset around {values["offset_median"]:.2f} seconds.
- Maximum signed offset around {values["offset_max"]:.2f} seconds.
- P75 around {values["offset_p75"]:.2f} seconds.
- P90 around {values["offset_p90"]:.2f} seconds.
- Approximately {values["percent_above_1_5"]:.1f}% of offsets are above 1.5 seconds.
""".strip()


def build_global_reference_section(
    profile,
):
    values = selected_shift_references[
        profile
    ]

    return f"""
{PROFILE_TITLES[profile]}:

- Mean best B correction shift is around {values["best_B_correction_shift_seconds__mean"]:.2f} seconds.
- Median best B correction shift is around {values["best_B_correction_shift_seconds__median"]:.2f} seconds.
- Mean estimated B lateness is around {values["estimated_B_lateness_seconds__mean"]:.2f} seconds.
- Median estimated B lateness is around {values["estimated_B_lateness_seconds__median"]:.2f} seconds.
- Mean alignment score gain versus zero shift is around {values["alignment_score_gain_vs_zero__mean"]:.3f}.
- Median alignment score gain versus zero shift is around {values["alignment_score_gain_vs_zero__median"]:.3f}.
- Mean number of bilateral alignment events is around {values["best_num_bilateral_events__mean"]:.2f}.
- Mean bilateral event coverage is around {100.0 * values["best_event_coverage__mean"]:.2f}%.
""".strip()


# Explicit order: NORMAL, LAG +2, LAG +3
DIAGNOSTIC_PROFILE_ORDER = [
    "NORMAL",
    "LAG_2",
    "LAG_3",
]


LOCAL_REFERENCE_TEXT = "\n\n".join(
    build_local_reference_section(
        profile
    )
    for profile in DIAGNOSTIC_PROFILE_ORDER
)


GLOBAL_REFERENCE_TEXT = "\n\n".join(
    build_global_reference_section(
        profile
    )
    for profile in DIAGNOSTIC_PROFILE_ORDER
)


# ============================================================
# EXACT DIAGNOSTIC 1 PROMPT TEMPLATE
# ============================================================

DIAGNOSTIC_PROMPT_TEMPLATE = r"""
You are evaluating temporal coordination between two participants in the same dyadic conversation.

You receive:

1. A signed strict A_end-to-B_start offset list.
2. Distribution summaries calculated from that exact signed list.
3. Global B-correction alignment-shift features.

SIGNED STRICT A_END-TO-B_START OFFSETS

Each value is calculated as:

B_start minus A_end

Positive value:

- Participant B starts after Participant A ends.
- Positive offsets are retained up to 6.0 seconds.
- Participant A must not start another turn before or at B_start.

Negative value:

- Participant B starts shortly before Participant A ends.
- Participant B must still be speaking when Participant A ends.
- Negative offsets are retained only when B starts during the current A turn and no more than 2.0 seconds before A_end.

Examples:

- -0.60 seconds means B starts 0.60 seconds before A ends.
- 0.00 seconds means an immediate boundary transition.
- 2.50 seconds means B starts 2.50 seconds after A ends.

The supplied list therefore describes local handoff timing on both sides of A_end.

SOFT REFERENCE PATTERNS FROM THE FROZEN 50-CONVERSATION REFERENCE SET

{local_reference_text}

These are soft exploratory patterns and not hard thresholds.
Natural variation exists in every profile.

DECISION GUIDANCE

- Use the complete signed offset distribution.
- Consider the full list, number of values, mean, median, maximum, P75, P90, and percentage above 1.5 seconds together.
- Compare the current case with the NORMAL profile and with each separate LAG +2 and LAG +3 profile.
- A later shift of Participant B tends to move signed offsets toward more positive values.
- Several elevated positive offsets support LAG.
- Elevated mean, median, P75, or P90 support LAG.
- Negative or near-zero offsets represent smooth or slightly overlapping handoffs and generally support NORMAL when they occur consistently.
- A single negative value does not automatically indicate NORMAL.
- A single positive maximum does not automatically indicate LAG.
- Do not decide from only one statistic.
- Do not require every value to be positive or large for LAG.
- A LAG case may still contain negative or short positive values.
- The final output remains binary. Do not predict the delay magnitude.

LIMITED EVIDENCE

- When the signed offset list is empty, the offset statistics are unavailable.
- When the list contains only one or two values, the statistics are based on limited evidence.
- In these cases, reduce confidence and use the global alignment-shift evidence cautiously.
- Do not classify LAG solely because of one large positive offset.

GLOBAL B-CORRECTION REFERENCE PATTERNS

The correction shift is a hypothetical global shift applied only during diagnostic alignment search.

{global_reference_text}

GLOBAL B-CORRECTION INTERPRETATION

- The best B correction shift is the global temporal correction that maximizes bilateral A-to-B and B-to-A boundary alignment.
- A negative correction means Participant B would need to move earlier to align better with Participant A.
- A correction near -2 seconds is compatible with an approximate +2 second Participant B delay.
- A correction near -3 seconds is compatible with an approximate +3 second Participant B delay.
- Estimated B lateness is max(0, -best correction shift).
- A correction near zero together with a very small alignment gain generally supports NORMAL.
- A clearly negative correction together with a meaningful alignment gain supports LAG.
- The correction value must not be used alone.
- The number of bilateral events and event coverage indicate how much evidence supports the correction estimate.
- A large correction based on very few events or very low coverage is weak evidence.
- Use the global-shift evidence together with the signed strict offset distribution.

LAG GENERATION

- A LAG case may contain Participant B shifted later by approximately 2 or 3 seconds.
- The delay magnitude, if any, is not provided for the current case.
- The signed offset list does not need to increase uniformly.
- Shifting Participant B can change which strict handoff events remain valid.

CLASSIFY AS NORMAL WHEN

- The signed offset distribution is predominantly negative, near zero, or short positive.
- The mean, median, P75, and P90 are more compatible with the NORMAL reference pattern than with the two LAG profiles.
- Values above 1.5 seconds are absent, rare, or isolated.
- A large maximum is not supported by the rest of the distribution.
- The global correction is near zero or is weakly supported.

CLASSIFY AS LAG WHEN

- The signed offset distribution is shifted toward positive values.
- Several offsets are elevated.
- Or the mean, median, P75, or P90 are more compatible with at least one of the separate LAG profiles.
- A meaningful percentage of offsets above 1.5 seconds supports LAG.
- A reliable negative global correction and meaningful alignment gain support LAG.

Use only the supplied temporal information.
Do not invent transcript content.
Do not use semantic assumptions.
Return one label even when the evidence is mixed.

Timeline duration:
{duration_seconds:.2f} seconds

DERIVED TEMPORAL FEATURES

Signed strict A_end-to-B_start offset list:
{signed_offsets}

Number of signed strict offsets:
{num_offsets}

Mean signed offset:
{offset_mean} seconds

Median signed offset:
{offset_median} seconds

Maximum signed offset:
{offset_max} seconds

P75 signed offset:
{offset_p75} seconds

P90 signed offset:
{offset_p90} seconds

Number of offsets above 1.5 seconds:
{num_above_1_5}

Percentage of offsets above 1.5 seconds:
{percent_above_1_5}%

GLOBAL ALIGNMENT-SHIFT FEATURES

Best B correction shift:
{best_shift} seconds

Estimated B lateness:
{estimated_lateness} seconds

Alignment score gain versus zero shift:
{alignment_gain}

Number of bilateral events supporting the best shift:
{num_bilateral_events}

Bilateral event coverage:
{event_coverage}%

Return ONLY valid JSON:
{{
  "label": "NORMAL or LAG",
  "confidence": 0.0,
  "reason": "brief explanation based on the signed offset distribution and global alignment-shift evidence"
}}
""".strip()


# ============================================================
# SAVE TEMPLATE
# ============================================================

PROMPT_TEMPLATE_PATH.write_text(
    DIAGNOSTIC_PROMPT_TEMPLATE,
    encoding="utf-8",
)


# ============================================================
# SAFE FORMATTERS
# ============================================================

def format_prompt_number(
    value,
    decimals,
):
    if value is None:
        return "unavailable"

    try:
        value = float(value)
    except (TypeError, ValueError):
        return "unavailable"

    if not math.isfinite(value):
        return "unavailable"

    return f"{value:.{decimals}f}"


def format_prompt_integer(
    value,
):
    if value is None:
        return "unavailable"

    try:
        value = float(value)
    except (TypeError, ValueError):
        return "unavailable"

    if not math.isfinite(value):
        return "unavailable"

    return str(
        int(
            round(value)
        )
    )


def format_prompt_offset_list(
    values,
):
    if values is None:
        return "[]"

    return json.dumps(
        list(values),
        ensure_ascii=False,
        separators=(",", ":"),
    )


# ============================================================
# BUILD EXACT MODEL-FACING PROMPT
# ============================================================

def build_diagnostic_prompt(
    case,
):
    payload = build_diagnostic_model_input(
        case
    )

    local_features = payload[
        "local_temporal_features"
    ]

    global_features = payload[
        "global_shift_features"
    ]

    prompt = DIAGNOSTIC_PROMPT_TEMPLATE.format(
        local_reference_text=LOCAL_REFERENCE_TEXT,
        global_reference_text=GLOBAL_REFERENCE_TEXT,

        duration_seconds=float(
            payload[
                "analysis_duration_seconds"
            ]
        ),

        signed_offsets=format_prompt_offset_list(
            local_features[
                "signed_strict_offsets_seconds"
            ]
        ),

        num_offsets=format_prompt_integer(
            local_features[
                "num_signed_strict_offsets"
            ]
        ),

        offset_mean=format_prompt_number(
            local_features[
                "offset_mean_seconds"
            ],
            2,
        ),

        offset_median=format_prompt_number(
            local_features[
                "offset_median_seconds"
            ],
            2,
        ),

        offset_max=format_prompt_number(
            local_features[
                "offset_max_seconds"
            ],
            2,
        ),

        offset_p75=format_prompt_number(
            local_features[
                "offset_p75_seconds"
            ],
            2,
        ),

        offset_p90=format_prompt_number(
            local_features[
                "offset_p90_seconds"
            ],
            2,
        ),

        num_above_1_5=format_prompt_integer(
            local_features[
                "num_offsets_above_1_5_seconds"
            ]
        ),

        percent_above_1_5=format_prompt_number(
            local_features[
                "percent_offsets_above_1_5_seconds"
            ],
            1,
        ),

        best_shift=format_prompt_number(
            global_features[
                "best_B_correction_shift_seconds"
            ],
            2,
        ),

        estimated_lateness=format_prompt_number(
            global_features[
                "estimated_B_lateness_seconds"
            ],
            2,
        ),

        alignment_gain=format_prompt_number(
            global_features[
                "alignment_score_gain_vs_zero"
            ],
            3,
        ),

        num_bilateral_events=format_prompt_integer(
            global_features[
                "best_num_bilateral_events"
            ]
        ),

        event_coverage=format_prompt_number(
            global_features[
                "best_event_coverage_percent"
            ],
            2,
        ),
    )

    prompt_lower = prompt.lower()

    forbidden_prompt_markers = [
        "turn format",
        "filtered turns:",
        "participant a filtered turns",
        "participant b filtered turns",
        "filtered overlap",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "typical lag +1",
        "correction near -1 second",
        "approximately 1, 2, or 3 seconds",
        "semantic summaries",
        "coarse_summary",
        "focused_summary",
        "gold_binary_label",
        "gold_anomaly_type",
        "case_id",
        "case_variant",
        "conversation_id",
        "participant_id",
        "reference_profile",
        "lag_seconds",
    ]

    for marker in forbidden_prompt_markers:
        assert marker not in prompt_lower, (
            f"Forbidden content leaked into prompt: {marker}"
        )

    required_prompt_markers = [
        "Typical NORMAL cases",
        "Typical LAG +2 second cases",
        "Typical LAG +3 second cases",
        "SIGNED STRICT A_END-TO-B_START OFFSETS",
        "GLOBAL B-CORRECTION REFERENCE PATTERNS",
        "GLOBAL B-CORRECTION INTERPRETATION",
        "CLASSIFY AS NORMAL WHEN",
        "CLASSIFY AS LAG WHEN",
        '"label": "NORMAL or LAG"',
    ]

    for marker in required_prompt_markers:
        assert marker in prompt, (
            f"Required prompt content is missing: {marker}"
        )

    return prompt


# Force a new inspection before inference.
DIAGNOSTIC_PROMPT_INSPECTED = False


print(
    "Prompt version:",
    DIAGNOSTIC_PROMPT_VERSION,
)

print(
    "Saved prompt template:",
    PROMPT_TEMPLATE_PATH,
)

print(
    "Prompt template characters:",
    len(
        DIAGNOSTIC_PROMPT_TEMPLATE
    ),
)

print(
    "Prompt inspection reset:",
    DIAGNOSTIC_PROMPT_INSPECTED,
)

Prompt version: diagnostic1_original_lag_prompt_R1_offsets_all_global_no_turns_no_overlap_v2
Saved prompt template: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/diagnostic1_targeted_lag_R1_temporal_features/prompt_template.txt
Prompt template characters: 6118
Prompt inspection reset: False


In [ ]:
print("Results path:", RESULTS_PATH)
print("Existing cache:", RESULTS_PATH.exists())

Results path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/diagnostic1_targeted_lag_R1_temporal_features/results_cache.json
Existing cache: False


## 7. Inspect the Exact Model-Facing Prompt

Before inference, one complete example is printed to audit the exact information available to Qwen2.5-Omni.

The inspection confirms that:

- the gold label is not included in the prompt,
- filtered turns are absent,
- overlap evidence is absent,
- participation evidence is absent,
- semantic evidence is absent,
- only the selected local and global temporal features are present,
- the frozen NORMAL / LAG +2 / LAG +3 reference profiles are available for comparison.

In [ ]:
inspection_case = next(
    case
    for case in diagnostic_cases
    if normalize_case_variant(
        case["case_variant"]
    ) == "lag_2sec"
)

inspection_payload = build_diagnostic_model_input(
    inspection_case
)

inspection_prompt = build_diagnostic_prompt(
    inspection_case
)

print("=" * 100)
print("DIAGNOSTIC 1 — EXACT PROMPT INSPECTION")
print("=" * 100)

print("Inspection case ID:", inspection_case["case_id"])
print("Gold label is intentionally NOT printed inside the model prompt.")
print("Filtered turns supplied: False")
print("Overlap evidence supplied: False")
print("Participant speaks supplied: False")
print("Semantic evidence supplied: False")
print("Local evidence: complete Structured R1 offset distribution.")
print("Global evidence: all five Structured R1 global features.")
print("Reference profiles: NORMAL, LAG +2, LAG +3.")

print("\n" + "=" * 100)
print("EXACT MODEL-FACING PAYLOAD")
print("=" * 100)

print(
    json.dumps(
        inspection_payload,
        indent=2,
        ensure_ascii=False,
    )
)

print("\n" + "=" * 100)
print("EXACT RENDERED MODEL PROMPT")
print("=" * 100)

print(
    inspection_prompt
)

DIAGNOSTIC_PROMPT_INSPECTED = True

print(
    "\nInspection flag set:",
    DIAGNOSTIC_PROMPT_INSPECTED,
)

DIAGNOSTIC 1 — EXACT PROMPT INSPECTION
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.
Filtered turns supplied: False
Overlap evidence supplied: False
Participant speaks supplied: False
Semantic evidence supplied: False
Local evidence: complete Structured R1 offset distribution.
Global evidence: all five Structured R1 global features.
Reference profiles: NORMAL, LAG +2, LAG +3.

EXACT MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "local_temporal_features": {
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offsets_above_1_5_seconds": 2,
    "percent_offsets_above_1_5_seconds": 100.0
  },
  "global_shift_features": {
    "best_B_correction_shift_seconds": 2.7,
    "estimated_B_latene

## 8. Load Qwen2.5-Omni Thinker

The same Qwen2.5-Omni-7B Thinker checkpoint used throughout the temporal reasoning experiments is loaded from Google Drive.

At this stage the model receives structured text evidence only; no raw audio or video is passed to the reasoner.

In [ ]:
import torch

from transformers import (
    Qwen2_5OmniThinkerForConditionalGeneration,
    Qwen2_5OmniProcessor,
)


print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU runtime before loading Qwen2.5-Omni-7B."
    )


print("GPU:", torch.cuda.get_device_name(0))

assert MODEL_ID == "Qwen/Qwen2.5-Omni-7B"


print("Loading model from:", MODEL_PATH)

model = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        str(MODEL_PATH),
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True,
    )
)

model.eval()

print("Loaded:", MODEL_ID)

`torch_dtype` is deprecated! Use `dtype` instead!


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Loading model from: /content/drive/MyDrive/Qwen2.5-Omni-7B


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-Omni-7B


In [ ]:
processor = Qwen2_5OmniProcessor.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    use_fast=False,
)

print("Processor loaded.")

Processor loaded.


## 9. JSON Parsing and Deterministic Text-Only Inference

The inference procedure preserves the original temporal-pipeline decoding policy.

Generation is deterministic (`do_sample=False`), and model responses are parsed into the required structured schema before evaluation.

In [ ]:
def extract_json_from_text(
    text,
):
    text = str(text).strip()

    text = re.sub(
        r"^```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    text = re.sub(
        r"```$",
        "",
        text,
    ).strip()

    try:
        return json.loads(
            text
        )

    except Exception:
        pass

    start = text.find(
        "{"
    )

    if start == -1:
        return {
            "parse_error": True,
            "raw_output": text,
        }

    depth = 0

    for index in range(
        start,
        len(text),
    ):
        if text[index] == "{":
            depth += 1

        elif text[index] == "}":
            depth -= 1

            if depth == 0:
                candidate = text[
                    start:index + 1
                ]

                try:
                    return json.loads(
                        candidate
                    )

                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {
        "parse_error": True,
        "raw_output": text,
    }


def normalize_lag_prediction(
    parsed,
):
    if not isinstance(
        parsed,
        dict,
    ):
        return None

    label = str(
        parsed.get(
            "label",
            "",
        )
    ).strip().upper()

    aliases = {
        "NORMAL": "NORMAL",
        "LAG": "LAG",
        "ANOMALOUS": "LAG",
        "ANOMALY": "LAG",
        "LAG ANOMALY": "LAG",
        "DELAY": "LAG",
        "DELAYED": "LAG",
    }

    return aliases.get(
        label
    )


def qwen_text_only(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {
        key: (
            value.to(model.device)
            if hasattr(value, "to")
            else value
        )
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[
            :,
            inputs["input_ids"].shape[1]:
        ],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        str(text).encode(
            "utf-8"
        )
    ).hexdigest()


def utc_now_iso():
    return datetime.now(
        timezone.utc
    ).isoformat()


def atomic_write_json(
    path,
    value,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )

## 10. Run Inference on the Exact 200 Cases

Qwen2.5-Omni is evaluated on all 200 fixed cases using the selected temporal representation.

Predictions are checkpointed to Google Drive after every completed case so that the experiment can resume safely after an interrupted Colab session.

The code and cache identifiers below retain the original `Diagnostic 1` naming from the run that generated the thesis result.

In [ ]:
assert (
    "DIAGNOSTIC_PROMPT_INSPECTED"
    in globals()
    and DIAGNOSTIC_PROMPT_INSPECTED
), (
    "Run the exact prompt-inspection cell before inference."
)


reference_hash = sha256_text(
    canonical_json({
        "base": selected_base_references,
        "shift": selected_shift_references,
    })
)


case_manifest_hash = sha256_text(
    canonical_json(
        case_manifest
    )
)


prompt_hash = sha256_text(
    DIAGNOSTIC_PROMPT_TEMPLATE
)


config_record = {
    "diagnostic_prompt_version": (
        DIAGNOSTIC_PROMPT_VERSION
    ),
    "model_id": MODEL_ID,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "prompt_hash": prompt_hash,
    "reference_hash": reference_hash,
    "case_manifest_hash": case_manifest_hash,
    "num_cases": len(diagnostic_cases),
    "local_fields": LOCAL_OFFSET_FIELDS,
    "global_fields": GLOBAL_MODEL_FIELDS,
}


config_hash = sha256_text(
    canonical_json(
        config_record
    )
)


def new_results_cache():
    return {
        "header": {
            **config_record,
            "config_hash": config_hash,
            "created_at_utc": utc_now_iso(),
            "updated_at_utc": utc_now_iso(),
        },
        "results": [],
    }


if RESULTS_PATH.exists():
    try:
        diagnostic_cache = load_json(
            RESULTS_PATH
        )

    except Exception as exc:
        raise RuntimeError(
            "Could not read the existing diagnostic cache. "
            "Move or repair the file before continuing."
        ) from exc

    assert isinstance(
        diagnostic_cache,
        dict,
    )

    assert (
        diagnostic_cache.get(
            "header",
            {},
        ).get(
            "config_hash"
        )
        == config_hash
    ), (
        "The existing cache was produced with a different prompt, "
        "reference set, case manifest, model configuration, or feature set. "
        "Use a new output directory or remove the incompatible cache."
    )

    assert isinstance(
        diagnostic_cache.get(
            "results"
        ),
        list,
    )

else:
    diagnostic_cache = new_results_cache()


compatible_results_by_case_id = {}


for result in diagnostic_cache[
    "results"
]:
    case_id = str(
        result.get(
            "case_id",
            "",
        )
    )

    prediction = normalize_lag_prediction(
        result.get(
            "parsed"
        )
    )

    if (
        case_id in set(diagnostic_case_ids)
        and prediction in {
            "NORMAL",
            "LAG",
        }
    ):
        compatible_results_by_case_id[
            case_id
        ] = result


cases_to_run = diagnostic_cases

if PILOT_MAX_CASES is not None:
    cases_to_run = cases_to_run[
        :PILOT_MAX_CASES
    ]


print("=" * 96)
print("DIAGNOSTIC 1 INFERENCE")
print("=" * 96)

print("Config hash:", config_hash)
print("Compatible cached predictions:", len(compatible_results_by_case_id))
print("Cases requested:", len(cases_to_run))
print(
    "Cases still missing:",
    sum(
        str(case["case_id"])
        not in compatible_results_by_case_id
        for case in cases_to_run
    ),
)


for case in tqdm(
    cases_to_run,
    desc="Diagnostic 1 NORMAL vs LAG +2/+3",
):
    case_id = str(
        case["case_id"]
    )

    if (
        case_id
        in compatible_results_by_case_id
    ):
        continue

    prompt = build_diagnostic_prompt(
        case
    )

    raw_output = qwen_text_only(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    parsed = extract_json_from_text(
        raw_output
    )

    payload = build_diagnostic_model_input(
        case
    )

    variant = normalize_case_variant(
        case["case_variant"]
    )

    reference_profile = {
        "normal": "NORMAL",
        "lag_2sec": "LAG_2",
        "lag_3sec": "LAG_3",
    }[
        variant
    ]

    gold_label = (
        "NORMAL"
        if variant == "normal"
        else "LAG"
    )

    result = {
        "case_id": case_id,

        # Evaluation-only metadata.
        # These fields never enter the model prompt.
        "case_variant": variant,
        "reference_profile": reference_profile,
        "gold_label": gold_label,

        "diagnostic_prompt_version": (
            DIAGNOSTIC_PROMPT_VERSION
        ),

        "config_hash": config_hash,
        "completed_at_utc": utc_now_iso(),

        "model_input": payload,
        "prompt": prompt,
        "raw_output": raw_output,
        "parsed": parsed,
    }

    diagnostic_cache[
        "results"
    ].append(
        result
    )

    compatible_results_by_case_id[
        case_id
    ] = result

    diagnostic_cache[
        "header"
    ][
        "updated_at_utc"
    ] = utc_now_iso()

    atomic_write_json(
        RESULTS_PATH,
        diagnostic_cache,
    )


print("\nTotal compatible stored predictions:", len(compatible_results_by_case_id))
print("Saved cache:", RESULTS_PATH)

DIAGNOSTIC 1 INFERENCE
Config hash: ab2ccb90599862a51fb606345eaeff8517b7ff5ad9fb60b6e9d74b3414a8bfe5
Compatible cached predictions: 0
Cases requested: 200
Cases still missing: 200


Diagnostic 1 NORMAL vs LAG +2/+3:   0%|          | 0/200 [00:00<?, ?it/s]


Total compatible stored predictions: 200
Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/diagnostic1_targeted_lag_R1_temporal_features/results_cache.json


## 11. Evaluate the Selected Temporal Representation

The final evaluation reports:

- valid and invalid prediction counts,
- exact schema rate,
- accuracy,
- balanced accuracy,
- classification report,
- binary confusion matrix,
- results separated by NORMAL, LAG +2 s, and LAG +3 s.

The reported result is:

- **NORMAL:** 83/100,
- **LAG +2 s:** 41/50,
- **LAG +3 s:** 47/50,
- **Overall:** 171/200 = **85.5%**.

This is the isolated temporal result carried into the thesis.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)


requested_case_ids = {
    str(case["case_id"])
    for case in cases_to_run
}


evaluation_results = {
    str(result["case_id"]): result
    for result in diagnostic_cache[
        "results"
    ]
    if (
        result.get(
            "config_hash"
        ) == config_hash
        and str(
            result.get(
                "case_id",
                "",
            )
        ) in requested_case_ids
    )
}


evaluation_rows = []


for case in cases_to_run:
    case_id = str(
        case["case_id"]
    )

    result = evaluation_results.get(
        case_id
    )

    if result is None:
        parsed = {}
        prediction = None
        raw_output = None
        prompt = None
        payload = build_diagnostic_model_input(
            case
        )

    else:
        parsed = result.get(
            "parsed",
            {},
        )

        prediction = normalize_lag_prediction(
            parsed
        )

        raw_output = result.get(
            "raw_output"
        )

        prompt = result.get(
            "prompt"
        )

        payload = result[
            "model_input"
        ]

    variant = normalize_case_variant(
        case["case_variant"]
    )

    reference_profile = {
        "normal": "NORMAL",
        "lag_2sec": "LAG_2",
        "lag_3sec": "LAG_3",
    }[
        variant
    ]

    gold_label = (
        "NORMAL"
        if variant == "normal"
        else "LAG"
    )

    local_features = payload[
        "local_temporal_features"
    ]

    global_features = payload[
        "global_shift_features"
    ]

    evaluation_rows.append({
        "case_id": case_id,
        "case_variant": variant,
        "reference_profile": reference_profile,
        "gold_label": gold_label,
        "prediction": prediction,
        "valid_prediction": prediction in {
            "NORMAL",
            "LAG",
        },
        "correct": prediction == gold_label,
        "confidence": (
            parsed.get("confidence")
            if isinstance(parsed, dict)
            else None
        ),
        "reason": (
            parsed.get("reason")
            if isinstance(parsed, dict)
            else None
        ),
        "raw_output": raw_output,
        "prompt": prompt,
        **local_features,
        **global_features,
    })


diagnostic_results_df = pd.DataFrame(
    evaluation_rows
)


diagnostic_results_df.to_csv(
    RESULTS_CSV_PATH,
    index=False,
)


valid_df = diagnostic_results_df[
    diagnostic_results_df[
        "valid_prediction"
    ]
].copy()


print("=" * 96)
print("DIAGNOSTIC 1 EVALUATION")
print("=" * 96)

print("Requested cases:", len(diagnostic_results_df))
print("Valid predictions:", len(valid_df))
print(
    "Invalid/missing predictions:",
    int(
        (
            ~diagnostic_results_df[
                "valid_prediction"
            ]
        ).sum()
    ),
)

print(
    "Exact schema/label rate:",
    round(
        diagnostic_results_df[
            "valid_prediction"
        ].mean(),
        4,
    ),
)

print("Saved CSV:", RESULTS_CSV_PATH)


if valid_df.empty:
    raise RuntimeError(
        "No valid NORMAL/LAG predictions were parsed."
    )


print(
    "\nAccuracy on valid predictions:",
    accuracy_score(
        valid_df["gold_label"],
        valid_df["prediction"],
    ),
)


print(
    "Balanced accuracy on valid predictions:",
    balanced_accuracy_score(
        valid_df["gold_label"],
        valid_df["prediction"],
    ),
)


strict_accuracy = (
    diagnostic_results_df[
        "correct"
    ].fillna(False).mean()
)


print(
    "Strict accuracy, invalid as wrong:",
    strict_accuracy,
)


print("\nCLASSIFICATION REPORT")

report_df = pd.DataFrame(
    classification_report(
        valid_df["gold_label"],
        valid_df["prediction"],
        labels=[
            "NORMAL",
            "LAG",
        ],
        output_dict=True,
        zero_division=0,
    )
).transpose()

display(
    report_df
)


cm = confusion_matrix(
    valid_df["gold_label"],
    valid_df["prediction"],
    labels=[
        "NORMAL",
        "LAG",
    ],
)


confusion_df = pd.DataFrame(
    cm,
    index=[
        "Gold NORMAL",
        "Gold LAG",
    ],
    columns=[
        "Pred NORMAL",
        "Pred LAG",
    ],
)


print("\nCONFUSION MATRIX")

display(
    confusion_df
)


print("\nPREDICTIONS BY TRUE PROFILE")

profile_prediction_table = (
    pd.crosstab(
        valid_df[
            "reference_profile"
        ],
        valid_df[
            "prediction"
        ],
    )
    .reindex(
        PROFILE_ORDER,
        fill_value=0,
    )
    .reindex(
        columns=[
            "NORMAL",
            "LAG",
        ],
        fill_value=0,
    )
)

profile_prediction_table.columns = [
    "Pred NORMAL",
    "Pred LAG",
]

display(
    profile_prediction_table
)


print("\nPER EXACT CASE VARIANT")

variant_rows = []

for variant in [
    "normal",
    "lag_2sec",
    "lag_3sec",
]:
    subset = diagnostic_results_df[
        diagnostic_results_df[
            "case_variant"
        ] == variant
    ]

    valid_subset = subset[
        subset[
            "valid_prediction"
        ]
    ]

    variant_rows.append({
        "case_variant": variant,
        "total_cases": len(subset),
        "valid_predictions": len(valid_subset),
        "invalid_predictions": int(
            (
                ~subset[
                    "valid_prediction"
                ]
            ).sum()
        ),
        "predicted_NORMAL": int(
            (
                valid_subset[
                    "prediction"
                ] == "NORMAL"
            ).sum()
        ),
        "predicted_LAG": int(
            (
                valid_subset[
                    "prediction"
                ] == "LAG"
            ).sum()
        ),
        "correct_predictions": int(
            subset[
                "correct"
            ].fillna(False).sum()
        ),
        "accuracy_on_valid": (
            valid_subset[
                "correct"
            ].mean()
            if len(valid_subset)
            else None
        ),
        "strict_accuracy_invalid_as_wrong": (
            subset[
                "correct"
            ].fillna(False).mean()
            if len(subset)
            else None
        ),
    })


variant_metrics_df = pd.DataFrame(
    variant_rows
)

display(
    variant_metrics_df
)

DIAGNOSTIC 1 EVALUATION
Requested cases: 200
Valid predictions: 200
Invalid/missing predictions: 0
Exact schema/label rate: 1.0
Saved CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/diagnostic1_targeted_lag_R1_temporal_features/results.csv

Accuracy on valid predictions: 0.855
Balanced accuracy on valid predictions: 0.855
Strict accuracy, invalid as wrong: 0.855

CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.873684,0.830,0.851282,100.000
LAG,0.838095,0.880,0.858537,100.000
accuracy,0.855000,0.855,0.855000,0.855
macro avg,0.855890,0.855,0.854909,200.000
weighted avg,0.855890,0.855,0.854909,200.000



CONFUSION MATRIX


,Pred NORMAL,Pred LAG
Gold NORMAL,83,17
Gold LAG,12,88



PREDICTIONS BY TRUE PROFILE


,Pred NORMAL,Pred LAG
reference_profile,,
NORMAL,83,17
LAG_2,9,41
LAG_3,3,47



PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_LAG,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong
0,normal,100,100,0,83,17,83,0.83,0.83
1,lag_2sec,50,50,0,9,41,41,0.82,0.82
2,lag_3sec,50,50,0,3,47,47,0.94,0.94


## 12. Error Inspection

The final cells expose the selected R1 local/global temporal evidence together with the model's brief explanation for correct and incorrect NORMAL/LAG subsets.

This section is diagnostic only; it does not modify predictions or contribute additional evidence to the classifier.

In [ ]:
CASE_DISPLAY_COLUMNS = [
    "case_id",
    "case_variant",
    "reference_profile",
    "gold_label",
    "prediction",
    "reason",
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage_percent",
]


def inspect_diagnostic_subset(
    title,
    gold_label,
    prediction,
):
    subset = diagnostic_results_df[
        (
            diagnostic_results_df[
                "gold_label"
            ] == gold_label
        )
        &
        (
            diagnostic_results_df[
                "prediction"
            ] == prediction
        )
    ].copy()

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    print("Cases:", len(subset))

    if subset.empty:
        return subset

    display(
        subset[
            CASE_DISPLAY_COLUMNS
        ]
        .sort_values(
            [
                "case_variant",
                "case_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return subset


diagnostic_normal_correct = inspect_diagnostic_subset(
    "NORMAL CASES CORRECTLY PREDICTED AS NORMAL",
    gold_label="NORMAL",
    prediction="NORMAL",
)


diagnostic_normal_false_positive = inspect_diagnostic_subset(
    "NORMAL CASES INCORRECTLY PREDICTED AS LAG",
    gold_label="NORMAL",
    prediction="LAG",
)


diagnostic_lag_correct = inspect_diagnostic_subset(
    "LAG CASES CORRECTLY PREDICTED AS LAG",
    gold_label="LAG",
    prediction="LAG",
)


diagnostic_lag_missed = inspect_diagnostic_subset(
    "LAG CASES INCORRECTLY PREDICTED AS NORMAL",
    gold_label="LAG",
    prediction="NORMAL",
)

## Optional: Disconnect the Colab Runtime

Run this only after inference, evaluation, inspection, and file saving are complete.

---

## Position in the Repository

Together, the two temporal notebooks document the complete experimental progression:

```text
Full temporal development pipeline
        ↓
richer temporal representation
(filtered turns + overlap + local offsets + global alignment)
        ↓
feature selection / simplification
        ↓
selected temporal representation
(local offset statistics + complete global alignment features)
        ↓
this notebook
        ↓
reported isolated temporal result: 85.5%
```

The development notebook explains **how the temporal branch was constructed and explored**.  
This notebook records **the reduced temporal configuration ultimately used for the thesis-reported isolated evaluation**.

In [ ]:
from google.colab import runtime

runtime.unassign()

# Final Result

The selected isolated temporal representation achieves **85.5% overall accuracy** on the exact balanced 200-case evaluation set.

Its class-specific performance is:

- **NORMAL:** 83/100
- **LAG +2 s:** 41/50
- **LAG +3 s:** 47/50

The increase from **82.0% at +2 s** to **94.0% at +3 s** is consistent with larger temporal perturbations being easier to distinguish from the frozen NORMAL reference profile.

Most importantly for the later unified system, this experiment demonstrates that useful LAG-specific information remains available after the richer temporal-development representation is reduced to **local response-offset statistics plus global alignment features**, without exposing filtered turn lists or overlap to the reasoner.